<a href="https://colab.research.google.com/github/ankita6876/CHATBOTS---Using-Natural-Language-Processing-and-Tensorflow/blob/main/CHATBOTS_Using_Natural_Language_Processing_and_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upgrade Version Trial

In [36]:
import json
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [37]:
with open('intents.json') as file:
    data = json.load(file)

In [38]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [39]:
sentences = []
labels_list = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        sentences.append(pattern)
        labels_list.append(intent["tag"])

In [40]:
labels = sorted(set(labels_list))
label_to_index = {label: index for index, label in enumerate(labels)}
indexed_labels = np.array([label_to_index[label] for label in labels_list])

In [41]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences)

In [42]:
from tensorflow.keras.layers import Embedding, LSTM

model = Sequential()

model.add(Embedding(
    input_dim=len(tokenizer.word_index) + 1,
    output_dim=16,
    input_length=padded.shape[1]
))

model.add(LSTM(32))

model.add(Dense(len(labels), activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [43]:
model.fit(padded, indexed_labels, epochs=300, verbose=1)

Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.0220 - loss: 2.3038  
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1685 - loss: 2.2987
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1979 - loss: 2.2941
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1607 - loss: 2.2904
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1763 - loss: 2.2853
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1763 - loss: 2.2777
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1490 - loss: 2.2756
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1841 - loss: 2.2667
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1802 - loss: 2.2552
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2154 - loss: 2.2419
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1685 - loss: 2.2359
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1958 - 

In [44]:
def chat():
    print("Bot is ready! Type 'quit' to exit.")

    context = None   # 🔥 memory variable

    while True:
        inp = input("You: ")

        if inp.lower() == "quit":
            break

        # Convert input to sequence
        seq = tokenizer.texts_to_sequences([inp])
        padded_seq = pad_sequences(seq, maxlen=padded.shape[1])

        result = model.predict(padded_seq)
        index = np.argmax(result)
        confidence = result[0][index]

        if confidence > 0.6:
            tag = labels[index]

            for intent in data["intents"]:
                if intent["tag"] == tag:

                    # 🔥 Check context_filter
                    if "context_filter" in intent:
                        if context != intent["context_filter"]:
                            continue

                    responses = intent["responses"]

                    # 🔥 Set new context if exists
                    if "context_set" in intent:
                        context = intent["context_set"]

                    print("Bot:", random.choice(responses))
                    break
        else:
            print("Bot: Sorry, I don't understand.")

In [45]:
model.save("chatbot_model.h5")

In [46]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [47]:
from google.colab import files
files.download("chatbot_model.h5")
files.download("tokenizer.pkl")
files.download("intents.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>